# 10x22 Vertical Dimensions

In [ ]:
EVENTS_RAW = {
    "David Robert Jones born": "1947-01-08",
    "Debut single 'Liza Jane' with Davie Jones & the King Bees": "1964-06-05",
    "Single 'Space Oddity' released": "1969-07-11",
    "Marries Angela Barnett": "1970-03-20",
    "Son Duncan Zowie Haywood Jones born": "1971-05-30",
    "Album 'Hunky Dory' released": "1971-12-17",
    "Comes out in a Melody Maker interview, declaring 'I’m gay'": "1972-01-22",
    "Album 'The Rise and Fall of Ziggy Stardust and the Spiders from Mars'": "1972-06-16",
    "Performs 'Starman' on BBC Top of the Pops": "1972-07-06",
    "Retires Ziggy Stardust onstage, Hammersmith Odeon": "1973-07-03",
    "Film 'The Man Who Fell to Earth' premieres": "1976-03-18",
    "Arrested in New York with Iggy Pop on marijuana charges": "1976-03-21",
    "Album 'Low' released": "1977-01-14",
    "Album 'Let's Dance' released": "1983-04-14",
    "Performs at Live Aid, Wembley Stadium": "1985-07-13",
    "Concert near the Berlin Wall": "1987-06-06",
    "Divorces Angela Barnett": "1980-02-08",
    "Bowie and Iman marry in a civil ceremony": "1992-04-24",
    "Wedding ceremony with Iman, Florence": "1992-06-06",
    "Inducted into Rock & Roll Hall of Fame": "1996-01-17",
    "Daughter Alexandria Zahra 'Lexi' Jones born": "2000-08-15",
    "Album 'Heathen' released": "2002-06-11",
    "Suffers an onstage heart attack in Germany, halting tour": "2004-06-25",
    "Album 'The Next Day' released": "2013-03-08",
    "Attends premiere of the musical Lazarus, his final public appearance": "2015-12-07",
    "Album 'Blackstar' released": "2016-01-08",
    "David Bowie dies (New York City)": "2016-01-10",
}

In [ ]:
# === David Bowie — Life in Days (Editable Text + Tilt/Spacing Jitter, Bounded Grid) ======
# 18x24 FULL-WIDTH + RANDOM WEEK WIDTHS (DIRICHLET) + HARD MIN WIDTH FLOOR (PRINT SAFE)
# TEXT: SINGLE LINE ONLY (no wrapping)

import sys, subprocess, os, math, time, datetime as dt

# -----------------------------------------------------------------------------------------
# Small helper to make sure required packages exist (good for Colab).
def _ensure(pkg):
    try:
        __import__(pkg.split("==")[0])
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

# Install deps
for _p in ["svgwrite", "cairosvg==2.7.0", "Pillow", "numpy"]:
    _ensure(_p)

import numpy as np
import svgwrite
from cairosvg import svg2png
from PIL import Image as PILImage
from IPython.display import Image as IPyImage, display

# -----------------------------------------------------------------------------------------
# CANVAS / COLORS

OUT_W_IN   = 18.0
OUT_H_IN   = 24.0
PRINT_DPI  = 300.0

VB_W = int(OUT_W_IN * PRINT_DPI)   # 5400 px
VB_H = int(OUT_H_IN * PRINT_DPI)   # 7200 px

BG_COLOR   = "#FFFFFF"

INK           = "#0B2530"
OUT_OF_LIFE   = "#C8D2D7"
EVENT_COLOR   = INK
EVENT_SW_MULT = 3.0

# TEXT STYLE
LABEL_FONT_SIZE_PX = 30.0
FONT_FAMILY = "Relief SingleLine Regular"
LINE_HEIGHT = 1.25  # kept but unused for single-line labels

# GRID
MARGIN   = 10.0
ROW_GAP  = 30.0
GAP_PX   = 3.0
CELL_INSET = 0.6

# HASHES
HASH_STROKE_PT       = "0.4pt"
VERT_TILT_DEG_MAX    = 2.2
HORIZ_TILT_DEG_MAX   = 1.2

# Spacing jitter inside each group (controls clumping)
SPACING_JITTER_FRAC  = 0.10   # tweak 0.06–0.14
CROSS_Y_RANGE_FRAC   = (0.42, 0.58)

# PRINT SAFETY: minimum width per calendar day in a week segment
MIN_PX_PER_DAY       = 12.0   # tweak 10–16

# Width randomness controls
DIRICHLET_ALPHA_RANGE = (0.45, 1.10)  # lower = more variation
DIRICHLET_PASSES      = 12

# -----------------------------------------------------------------------------------------
# DATES & EVENTS
BIRTH = dt.date(1947, 1, 8)
DEATH = dt.date(2016, 1, 10)

EVENTS_RAW = {
    "David Robert Jones born": "1947-01-08",
    "Debut single 'Liza Jane' with Davie Jones & the King Bees": "1964-06-05",
    "Single 'Space Oddity' released": "1969-07-11",
    "Marries Angela Barnett": "1970-03-20",
    "Son Duncan Zowie Haywood Jones born": "1971-05-30",
    "Album 'Hunky Dory' released": "1971-12-17",
    "Comes out in a Melody Maker interview, declaring 'I’m gay'": "1972-01-22",
    "Album 'The Rise and Fall of Ziggy Stardust and the Spiders from Mars'": "1972-06-16",
    "Performs 'Starman' on BBC Top of the Pops": "1972-07-06",
    "Retires Ziggy Stardust onstage, Hammersmith Odeon": "1973-07-03",
    "Film 'The Man Who Fell to Earth' premieres": "1976-03-18",
    "Arrested in New York alongside Iggy Pop on marijuana charges": "1976-03-21",
    "Album 'Low' released": "1977-01-14",
    "Album 'Let's Dance' released": "1983-04-14",
    "Performs at Live Aid, Wembley Stadium": "1985-07-13",
    "Concert near the Berlin Wall": "1987-06-06",
    "Divorces Angela Barnett": "1980-02-08",
    "Bowie and Iman marry in a civil ceremony": "1992-04-24",
    "Wedding ceremony with Iman, Florence": "1992-06-06",
    "Inducted into Rock & Roll Hall of Fame": "1996-01-17",
    "Daughter Alexandria Zahra 'Lexi' Jones born": "2000-08-15",
    "Album 'Heathen' released": "2002-06-11",
    "Suffers an onstage heart attack in Germany, halting tour": "2004-06-25",
    "Album 'The Next Day' released": "2013-03-08",
    "Attends premiere of the musical Lazarus, his final public appearance": "2015-12-07",
    "Album 'Blackstar' released": "2016-01-08",
    "David Bowie dies (New York City)": "2016-01-10",
}

def _to_date(s):
    y, m, d = map(int, s.split("-"))
    return dt.date(y, m, d)

def _day_index(d):
    return (d - BIRTH).days

TOTAL_DAYS = (DEATH - BIRTH).days

EVENTS = {}
for order_idx, (title, datestr) in enumerate(EVENTS_RAW.items()):
    d = _to_date(datestr)
    if BIRTH <= d <= DEATH:
        di = _day_index(d)
        EVENTS.setdefault(di, []).append({
            "date": d.isoformat(),
            "title": title,
            "order": order_idx,
        })

# -----------------------------------------------------------------------------------------
# DATE HELPERS
def year_start(y): return dt.date(y, 1, 1)
def year_end(y):   return dt.date(y, 12, 31)

YEAR_WEEK_RANGES = {}

def _build_year_week_ranges():
    for y in range(BIRTH.year, DEATH.year + 1):
        ys, ye = year_start(y), year_end(y)
        segs = []
        cur = ys
        one = dt.timedelta(days=1)
        while cur <= ye:
            wd = cur.weekday()  # Monday=0, ..., Sunday=6
            if wd == 6:
                d_end = min(ye, cur + dt.timedelta(days=6))
            else:
                days_until_sunday = 6 - wd
                d_end = min(ye, cur + dt.timedelta(days=days_until_sunday - 1))
            segs.append((cur, d_end))
            cur = d_end + one
        YEAR_WEEK_RANGES[y] = segs

_build_year_week_ranges()

def days_for_year_week(y, week_idx_0_based):
    segs = YEAR_WEEK_RANGES.get(y, [])
    if not (0 <= week_idx_0_based < len(segs)):
        return []
    d0, d1 = segs[week_idx_0_based]
    cur = d0
    one = dt.timedelta(days=1)
    ret = []
    while cur <= d1:
        if BIRTH <= cur <= DEATH:
            ret.append(_day_index(cur))
        cur += one
    return ret

# -----------------------------------------------------------------------------------------
# LAYOUT HELPERS
def compute_row_heights(n_rows, inner_height, row_gap, rng,
                        chaos_row_height_factor_range=(0.7, 1.3),
                        min_row_h_px=6.0):
    total_gaps = row_gap*(n_rows-1) if n_rows > 1 else 0.0
    avail = max(0.0, inner_height - total_gaps)
    lo, hi = chaos_row_height_factor_range
    ws = [rng.uniform(lo, hi) for _ in range(n_rows)]
    base = min_row_h_px*n_rows
    rem = max(0.0, avail - base)
    s = sum(ws) or n_rows
    hs = [min_row_h_px + rem*(w/s) for w in ws]
    scale = avail / max(1e-9, sum(hs))
    return [h*scale for h in hs]

def random_week_widths_with_floor(year, total_width_px, min_px_per_day=12.0,
                                 alpha_range=(0.45, 1.10), passes=12, rng=None):
    rng = rng or np.random.default_rng()
    segs = YEAR_WEEK_RANGES[year]
    n = len(segs)

    day_counts = np.array([(d1 - d0).days + 1 for (d0, d1) in segs], dtype=float)
    floor = day_counts * float(min_px_per_day)

    alphas = rng.uniform(alpha_range[0], alpha_range[1], size=n)
    w = rng.gamma(shape=alphas, scale=1.0)
    w = w / (w.sum() or 1.0)
    widths = w * total_width_px

    for _ in range(max(1, int(passes))):
        widths = np.maximum(widths, floor)
        s = float(widths.sum() or 1.0)
        widths *= (total_width_px / s)

    widths[-1] += (total_width_px - float(widths.sum()))
    return widths.tolist()

# -----------------------------------------------------------------------------------------
# SINGLE-LINE LABELS ONLY (no wrapping)
def label_to_single_line(label: str) -> str:
    return " ".join(str(label).split())

# -----------------------------------------------------------------------------------------
# HASH DRAWING
def _scale_stroke_width(sw_value, mult):
    if isinstance(sw_value, (int, float)):
        return float(sw_value) * float(mult)
    s = str(sw_value).strip()
    try:
        v = float(s); return v * float(mult)
    except Exception:
        pass
    if s.endswith("pt"):
        v = float(s[:-2]); return f"{v * float(mult)}pt"
    if s.endswith("px"):
        v = float(s[:-2]); return f"{v * float(mult)}px"
    return s

def _jittered_vertical_xs(rng, xi, wi, n_lines=6, jitter_frac=SPACING_JITTER_FRAC):
    if n_lines <= 0 or wi <= 0:
        return []
    n_gaps = n_lines + 1
    base = rng.uniform(1.0 - jitter_frac, 1.0 + jitter_frac, size=n_gaps)
    base = np.maximum(0.12, base)
    gap_w = wi * (base / base.sum())

    xs = []
    cur_x = xi + gap_w[0]
    for i in range(n_lines):
        xs.append(cur_x)
        if i + 1 < n_gaps:
            cur_x += gap_w[i+1]
    return xs

def _event_targets(days_in_cell, day_weekdays, n_vertical, has_saturday):
    evs = []
    for pos, day_abs in enumerate(days_in_cell):
        if day_abs in EVENTS:
            for ev in EVENTS[day_abs]:
                evs.append((pos, day_abs, ev["order"]))
    if not evs:
        return set(), False

    evs.sort(key=lambda t: (t[0], t[2]))
    first_pos, _, _ = evs[0]
    wd = day_weekdays[first_pos] if first_pos < len(day_weekdays) else None

    if wd == 5 and has_saturday:
        return set(), True

    if n_vertical <= 0:
        return set(), False

    pos_to_vert = {}
    v_idx = 0
    for pos, wd_i in enumerate(day_weekdays):
        if wd_i == 5:
            continue
        pos_to_vert[pos] = v_idx
        v_idx += 1

    if first_pos not in pos_to_vert:
        return set(), False

    return {pos_to_vert[first_pos]}, False

def draw_week_hashes_tilted(
    rng,
    dwg_group,
    x, y, w, h,
    days_in_cell,
    stroke=INK,
    stroke_width_pt=HASH_STROKE_PT,
    event_color=EVENT_COLOR,
    inset_px=CELL_INSET
):
    if w <= 0 or h <= 0:
        return
    xi = x + inset_px
    yi = y + inset_px
    wi = max(0.0, w - 2*inset_px)
    hi = max(0.0, h - 2*inset_px)
    if wi <= 0 or hi <= 0:
        return

    if days_in_cell:
        day_weekdays = []
        for dabs in days_in_cell:
            d_date = BIRTH + dt.timedelta(days=int(dabs))
            day_weekdays.append(d_date.weekday())

        has_saturday = any(wd == 5 for wd in day_weekdays)
        n_vertical = sum(1 for wd in day_weekdays if wd != 5)  # keep mark count
        event_vset, cross_is_event = _event_targets(days_in_cell, day_weekdays, n_vertical, has_saturday)
    else:
        has_saturday = True
        n_vertical = 6
        event_vset, cross_is_event = set(), False

    xs = _jittered_vertical_xs(rng, xi, wi, n_lines=n_vertical, jitter_frac=SPACING_JITTER_FRAC) if n_vertical > 0 else []

    for k, x_center in enumerate(xs, start=0):
        theta_deg = 90.0 + rng.uniform(-VERT_TILT_DEG_MAX, VERT_TILT_DEG_MAX)
        m = math.tan(math.radians(theta_deg - 90.0))

        height_jitter = rng.uniform(-VERT_HEIGHT_JITTER_FRAC, VERT_HEIGHT_JITTER_FRAC) * hi
        y_jitter = rng.uniform(-VERT_Y_JITTER_PX, VERT_Y_JITTER_PX)

        local_hi = max(hi * 0.4, hi + height_jitter)

        y_top = yi + y_jitter
        y_bot = y_top + local_hi
        y_mid = (y_top + y_bot) / 2.0

        x_top = x_center + m * (y_top - y_mid)
        x_bot = x_center + m * (y_bot - y_mid)

        is_event_vertical = (k in event_vset)
        sw_value = _scale_stroke_width(stroke_width_pt, EVENT_SW_MULT) if is_event_vertical else stroke_width_pt
        p = svgwrite.path.Path(
            d=f"M {x_top:.3f},{y_top:.3f} L {x_bot:.3f},{y_bot:.3f}",
            fill="none", stroke=stroke, stroke_width=sw_value
        )
        p.attribs["stroke-linecap"] = "round"
        p.attribs["stroke-linejoin"] = "round"
        p.attribs["style"] = "vector-effect:non-scaling-stroke"
        dwg_group.add(p)

    if has_saturday:
        frac = rng.uniform(*CROSS_Y_RANGE_FRAC)
        yc   = yi + frac * hi
        phi_deg = rng.uniform(-HORIZ_TILT_DEG_MAX, HORIZ_TILT_DEG_MAX)
        t = math.tan(math.radians(phi_deg))

        x_mid = xi + wi/2.0
        y_left  = yc + t * (xi - x_mid)
        y_right = yc + t * ((xi + wi) - x_mid)

        h_sw = _scale_stroke_width(stroke_width_pt, EVENT_SW_MULT) if cross_is_event else stroke_width_pt
        hline = svgwrite.path.Path(
            d=f"M {xi:.3f},{y_left:.3f} L {xi + wi:.3f},{y_right:.3f}",
            fill="none", stroke=stroke, stroke_width=h_sw
        )
        hline.attribs["stroke-linecap"] = "round"
        hline.attribs["stroke-linejoin"] = "round"
        hline.attribs["style"] = "vector-effect:non-scaling-stroke"
        dwg_group.add(hline)

# -----------------------------------------------------------------------------------------
def draw_cash_weeks_grid_with_editable_text(
    seed=123,
    chaos_row_height_factor_range=(0.9, 1.1),
    min_row_h_px=10.0,
    show_out_of_life=True,
    label_pos="top"
):
    rng = np.random.default_rng(seed)

    years = list(range(BIRTH.year, DEATH.year + 1))
    n_rows = len(years)

    ts = time.strftime("%Y%m%d_%H%M%S")
    name = f"cash_life_editable_text_tilt_spacing_{seed}_{ts}"
    out_dir = "/content"; os.makedirs(out_dir, exist_ok=True)
    svg_path = os.path.join(out_dir, f"{name}.svg")

    dwg = svgwrite.Drawing(
        svg_path,
        size=(f"{OUT_W_IN}in", f"{OUT_H_IN}in"),
        viewBox=f"0 0 {VB_W} {VB_H}"
    )
    dwg.add(dwg.rect(insert=(0,0), size=(VB_W, VB_H), fill=BG_COLOR))

    g_lines = svgwrite.container.Group(id="weeks", fill="none", stroke=INK)
    g_lines.attribs["style"] = "vector-effect:non-scaling-stroke"
    g_text  = svgwrite.container.Group(id="labels", fill="none", stroke="none")

    inner_L = MARGIN; inner_R = VB_W - MARGIN
    inner_T = MARGIN; inner_B = VB_H - MARGIN
    inner_W = inner_R - inner_L; inner_H = inner_B - inner_T

    row_heights = compute_row_heights(
        n_rows=n_rows, inner_height=inner_H, row_gap=ROW_GAP,
        rng=np.random.default_rng(seed),
        chaos_row_height_factor_range=chaos_row_height_factor_range,
        min_row_h_px=min_row_h_px
    )

    y_cursor = inner_T
    for ri, y in enumerate(years):
        y0 = y_cursor
        h  = row_heights[ri]

        cols = len(YEAR_WEEK_RANGES[y])
        if cols == 0:
            continue

        total_gaps = GAP_PX * (cols - 1) if cols > 1 else 0.0
        avail_w_for_weeks = max(1.0, inner_W - total_gaps)

        base_week_widths = random_week_widths_with_floor(
            year=y,
            total_width_px=avail_w_for_weeks,
            min_px_per_day=MIN_PX_PER_DAY,
            alpha_range=DIRICHLET_ALPHA_RANGE,
            passes=DIRICHLET_PASSES,
            rng=np.random.default_rng(seed + ri)
        )

        segments = []
        for ci in range(cols):
            days_in_cell = days_for_year_week(y, ci)
            base_hash_w = base_week_widths[ci]

            cell_events = []
            for pos, dabs in enumerate(days_in_cell):
                if dabs in EVENTS:
                    for ev in EVENTS[dabs]:
                        cell_events.append((pos, dabs, ev["order"], ev["title"]))

            if days_in_cell and cell_events:
                cell_events.sort(key=lambda t: (t[0], t[2]))

                if len(cell_events) == 1:
                    event_pos, _, _, title = cell_events[0]
                    n_days = len(days_in_cell)
                    if n_days == 7 and event_pos == 6:
                        event_hash_idx = 7
                    else:
                        event_hash_idx = int(math.floor(event_pos * 6.0 / max(1, n_days))) + 1

                    label = label_to_single_line(title)
                    est_text_px = len(label) * CHAR_WIDTH_PX
                    place_before = (1 <= event_hash_idx <= 3)

                    if place_before:
                        segments.append({"kind":"text","width":est_text_px,"label":label,
                                         "year":y,"week_idx":ci,"days_in_cell":days_in_cell})
                        segments.append({"kind":"gap","width":GAP_PX})
                        segments.append({"kind":"hash","width":base_hash_w,"year":y,"week_idx":ci,
                                         "days_in_cell":days_in_cell,"stroke": INK if days_in_cell else OUT_OF_LIFE})
                    else:
                        segments.append({"kind":"hash","width":base_hash_w,"year":y,"week_idx":ci,
                                         "days_in_cell":days_in_cell,"stroke": INK if days_in_cell else OUT_OF_LIFE})
                        segments.append({"kind":"gap","width":GAP_PX})
                        segments.append({"kind":"text","width":est_text_px,"label":label,
                                         "year":y,"week_idx":ci,"days_in_cell":days_in_cell})

                else:
                    first_title = cell_events[0][3]
                    last_title  = cell_events[-1][3]

                    first_label = label_to_single_line(first_title)
                    last_label  = label_to_single_line(last_title)

                    est_first_px = len(first_label) * CHAR_WIDTH_PX
                    est_last_px  = len(last_label)  * CHAR_WIDTH_PX

                    segments.append({"kind":"text","width":est_first_px,"label":first_label,
                                     "year":y,"week_idx":ci,"days_in_cell":days_in_cell})
                    segments.append({"kind":"gap","width":GAP_PX})
                    segments.append({"kind":"hash","width":base_hash_w,"year":y,"week_idx":ci,
                                     "days_in_cell":days_in_cell,"stroke": INK if days_in_cell else OUT_OF_LIFE})
                    segments.append({"kind":"gap","width":GAP_PX})
                    segments.append({"kind":"text","width":est_last_px,"label":last_label,
                                     "year":y,"week_idx":ci,"days_in_cell":days_in_cell})

            else:
                segments.append({"kind":"hash","width":base_hash_w,"year":y,"week_idx":ci,
                                 "days_in_cell":days_in_cell,"stroke": INK if days_in_cell else OUT_OF_LIFE})

            if ci < cols - 1:
                segments.append({"kind":"gap","width":GAP_PX})

        total_text_w = sum(s["width"] for s in segments if s["kind"] == "text")
        total_hash_w = sum(s["width"] for s in segments if s["kind"] == "hash")
        total_gap_w  = sum(s["width"] for s in segments if s["kind"] == "gap")
        row_total_w  = total_text_w + total_hash_w + total_gap_w

        if total_hash_w > 0 and row_total_w > inner_W:
            space_for_hashes = inner_W - (total_text_w + total_gap_w)
            hash_scale = max(0.01, space_for_hashes / total_hash_w)
        else:
            hash_scale = 1.0

        x_cursor = inner_L
        for seg in segments:
            kind = seg["kind"]
            w_seg = seg["width"]

            if kind == "hash":
                final_w = w_seg * hash_scale
                draw_week_hashes_tilted(
                    rng, g_lines,
                    x_cursor, y0, final_w, h,
                    days_in_cell=seg["days_in_cell"],
                    stroke=seg["stroke"],
                    inset_px=CELL_INSET
                )
                x_cursor += final_w

            elif kind == "text":
                final_w = w_seg
                yi = y0 + CELL_INSET
                hi = max(0.0, h - 2*CELL_INSET)

                baseline_y0 = (yi + hi / 2.0) if label_pos == "center" else (yi + LABEL_FONT_SIZE_PX + 1.0)
                text_cx = x_cursor + final_w / 2.0
                label = seg.get("label", "")

                txt = dwg.text(
                    label,
                    insert=(text_cx, baseline_y0),
                    fill=INK,
                    text_anchor="middle",
                    font_size=f"{LABEL_FONT_SIZE_PX}px",
                    font_family=FONT_FAMILY
                )
                g_text.add(txt)
                x_cursor += final_w

            else:
                x_cursor += w_seg

        y_cursor += h
        if ri < n_rows - 1:
            y_cursor += ROW_GAP

    dwg.add(g_lines)
    dwg.add(g_text)
    dwg.save()
    print("SVG saved:", svg_path)

    png_path = svg_path.replace(".svg", "_preview_300dpi.png")
    jpg_path = svg_path.replace(".svg", "_preview_300dpi.jpg")

    svg2png(url=svg_path, write_to=png_path, output_width=VB_W, output_height=VB_H, background_color="white")
    im = PILImage.open(png_path).convert("RGBA")
    white = PILImage.new("RGB", im.size, (255, 255, 255))
    white.paste(im, mask=im.split()[-1])
    white.save(jpg_path, "JPEG", quality=85, optimize=True)

    display(IPyImage(filename=jpg_path, width=min(VB_W, 1000)))
    print("Preview JPG:", jpg_path)
    return svg_path

# -----------------------------------------------------------------------------------------
# TWEAKABLE PARAMS
SHOW_DAY_LABEL         = False

CHAR_WIDTH_MM          = 1.5
MM_TO_PX               = PRINT_DPI / 25.4
CHAR_WIDTH_PX          = CHAR_WIDTH_MM * MM_TO_PX

VERT_HEIGHT_JITTER_FRAC = 0.12
VERT_Y_JITTER_PX        = 1.8

# -----------------------------------------------------------------------------------------
# RUN
_ = draw_cash_weeks_grid_with_editable_text(
    seed=999,
    chaos_row_height_factor_range=(0.9, 1.1),
    min_row_h_px=10.0,
    show_out_of_life=True,
    label_pos="top"
)


- Have thicker lines actually be 3-4 same size lines right next to each other